In [1]:
import pandas as pd
import xarray as xr
from matplotlib import pyplot as plt
import xarray as xr
from scipy.stats import wasserstein_distance
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import sys

sys.path.insert(
    1, "/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam/evaluation"
)
import plot_utils as utils
import numpy as np

sys.path.insert(
    1, "/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam"
)

from neural_lam.configs import get_constants

c = get_constants("ukesm")

# future_models = utils.FUTURE_MODELS_UKESM[:-2] + [utils.FUTURE_MODELS_UKESM[-1]]

# past_models = utils.PAST_MODELS_UKESM[:-2] + [utils.PAST_MODELS_UKESM[-1]]

future_models = utils.FUTURE_MODELS_UKESM

past_models = utils.PAST_MODELS_UKESM


model_resolution = [
    "z64",
    "z128",
    "z256",
    "z64 hierarchical",
    "persistence",
]

for idx, model in enumerate(future_models):
    model["rmse"] = pd.read_csv(
        f"/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam/eval_ukesm/rmse_r2_final/rmse_all_variables_{model['id']}.csv",
        index_col=0,
    )
    model["r2"] = pd.read_csv(
        f"/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam/eval_ukesm/rmse_r2_final/r2_all_variables_{model['id']}.csv",
        index_col=0,
    )
for idx, model in enumerate(past_models):
    model["rmse"] = pd.read_csv(
        f"/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam/eval_ukesm/rmse_r2_final/rmse_all_variables_{model['id']}.csv",
        index_col=0,
    )
    model["r2"] = pd.read_csv(
        f"/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam/eval_ukesm/rmse_r2_final/r2_all_variables_{model['id']}.csv",
        index_col=0,
    )

all_vars = utils.KEY_VARIABLES_UKESM

# Compute the mean MAE across all 83 variables (columns) for each time step (row)


plot_folder = "/home/hk-project-pai00005/xo8179/neural_lam_fork/neural-lam/eval_ukesm/plots"


figname = "rmse_allmodels_future"

# Plot RMSE + R2

In [2]:
import matplotlib.lines as mlines
model_short_names = ['UKESM z64-MM','UKESM z128-MM','UKESM z256-MM','UKESM z64-H', 'Persistence']
lw=1
for display_name, unit, variable, plevel, _ in all_vars:
    figsize = (9,3)
    if 'Wind' in display_name:
        figsize = (9,3.2)
        display_name=display_name+'\n'
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    for idx,model in enumerate(past_models):
        r2 = model["r2"].loc[variable]
        axes[0].plot(
            [x for x in range(1, 11)],
            r2.groupby(np.arange(len(r2)) // 4).mean(),
            label=model_short_names[idx],
            linewidth=lw,
        )

    legend1 = axes[0].legend(title="Models", loc="lower left", framealpha=0.6)
    axes[0].add_artist(legend1)
    for idx, model in enumerate(future_models):
        r2 = model["r2"].loc[variable]
        axes[0].plot(
            [x for x in range(1, 11)],
            r2.groupby(np.arange(len(r2)) // 4).mean(),
            label=model_short_names[idx] + f" Future",
            linestyle="dashed",
            linewidth=lw,
            color=f"C{idx}",
        )

    # Add line style legend for past vs future
    past_line = mlines.Line2D([], [], color="black", linestyle="-", label="U_T1")
    future_line = mlines.Line2D(
        [], [], color="black", linestyle="--", label="U_T2"
    )
    axes[0].legend(
        handles=[past_line, future_line], loc="upper right", title="Test Set", framealpha=0.6
    )

    axes[0].set_title(f"{display_name} - R² Daily Average")
    axes[0].set_xlabel("Forecasting Time [Days]")
    axes[0].set_ylabel(f"R² score")
    axes[0].grid(True)
    
    
    for idx,model in enumerate(past_models):
        rmse = model["rmse"].loc[variable]
        axes[1].plot(
            [x for x in range(1, 11)],
            rmse.groupby(np.arange(len(rmse)) // 4).mean(),
            linewidth=lw,
            label=model_short_names[idx]
        )

    for idx, model in enumerate(future_models):
        rmse = model["rmse"].loc[variable]
        axes[1].plot(
            [x for x in range(1, 11)],
            rmse.groupby(np.arange(len(rmse)) // 4).mean(),
            label=model_short_names[idx] + f" Future",
            linestyle="dashed",
            linewidth=lw,
            color=f"C{idx}",
        )


    axes[1].set_title(f"{display_name} - RMSE Daily Average")
    axes[1].set_xlabel("Forecasting Time [Days]")
    axes[1].set_ylabel(f"RMSE")
    axes[1].grid(True)
    fig.tight_layout()
    fig.savefig(
        f"{plot_folder}/rmse_r2_plots/UKESM_daily_average_{figname}_{variable}.png",
        dpi=400,
    )
    fig.savefig(
        f"{plot_folder}/rmse_r2_plots/UKESM_daily_average_{figname}_{variable}.pdf",
        dpi=400,
    )
    plt.close()

# Plot all, limit y

In [8]:
import matplotlib.lines as mlines
model_short_names = ['UKESM z64-MM','UKESM z128-MM','UKESM z256-MM','UKESM z64-H', 'Persistence']
lw=1
for display_name, unit, variable, plevel, _ in [all_vars[1]]:

    fig, axes = plt.subplots(1, 2, figsize=(9,3))
    lowest_y=1
    for idx,model in enumerate(past_models):
        r2 = model["r2"].loc[variable]
        data=r2.groupby(np.arange(len(r2)) // 4).mean()
        axes[0].plot(
            [x for x in range(1, 11)],
            data,
            linewidth=lw,
            label=model_short_names[idx]
        )
        if idx>0:
            if min(data) < lowest_y:
                lowest_y = min(data)

    legend1 = axes[0].legend(title="Models", loc="lower left", framealpha=0.6)
    axes[0].add_artist(legend1)
    for idx, model in enumerate(future_models):
        r2 = model["r2"].loc[variable]
        data=r2.groupby(np.arange(len(r2)) // 4).mean()
        axes[0].plot(
            [x for x in range(1, 11)],
            data,
            label=model_short_names[idx] + f" Future",
            linestyle="dashed",
            linewidth=lw,
            color=f"C{idx}",
        )
        if idx>0:
            if min(data) < lowest_y:
                lowest_y = min(data)

    # Add line style legend for past vs future
    past_line = mlines.Line2D([], [], color="black", linestyle="-", label="U_T1")
    future_line = mlines.Line2D(
        [], [], color="black", linestyle="--", label="U_T2"
    )
    axes[0].legend(
        handles=[past_line, future_line], loc="upper right", title="Test Set", framealpha=0.6
    )

    axes[0].set_title(f"{display_name} - R² Daily Average")
    axes[0].set_xlabel("Forecasting Time [Days]")
    axes[0].set_ylabel(f"R² score")
    axes[0].grid(True)
    axes[0].set_ylim((lowest_y,1))
    
    highest_y=0
    for idx,model in enumerate(past_models):
        rmse = model["rmse"].loc[variable]
        data=rmse.groupby(np.arange(len(rmse)) // 4).mean()
        axes[1].plot(
            [x for x in range(1, 11)],
            data,
            linewidth=lw,
            label=model_short_names[idx]
        )
        if idx>0:
            if max(data) > highest_y:
                highest_y = max(data)

    for idx, model in enumerate(future_models):
        rmse = model["rmse"].loc[variable]
        data=rmse.groupby(np.arange(len(rmse)) // 4).mean()
        axes[1].plot(
            [x for x in range(1, 11)],
            data,
            label=model_short_names[idx] + f" Future",
            linestyle="dashed",
            linewidth=lw,
            color=f"C{idx}",
        )
        if idx>0:
            if max(data) > highest_y:
                highest_y = max(data)

    axes[1].set_ylim((0,highest_y))

    axes[1].set_title(f"{display_name} - RMSE Daily Average")
    axes[1].set_xlabel("Forecasting Time [Days]")
    axes[1].set_ylabel(f"RMSE")
    axes[1].grid(True)
    fig.tight_layout()
    fig.savefig(
        f"{plot_folder}/rmse_r2_plots/UKESM_daily_average_{figname}_{variable}_limited.png",
        dpi=400,
    )
    fig.savefig(
        f"{plot_folder}/rmse_r2_plots/UKESM_daily_average_{figname}_{variable}_limited.pdf",
        dpi=400,
    )
    plt.close()

# Plot RMSE

In [ ]:
import matplotlib.lines as mlines

for display_name, unit, variable, plevel, _ in all_vars:

    fig, axes = plt.subplots(1, 1, figsize=(6,4))
    
    for model in past_models:
        rmse = model["rmse"].loc[variable]
        axes.plot(
            [x for x in range(1, 11)],
            rmse.groupby(np.arange(len(rmse)) // 4).mean(),
            label=model["model_name"] + f"",
        )

    legend1 = axes.legend(
        title="Models",
        loc="lower right",
    )
    axes.add_artist(legend1)
    for idx, model in enumerate(future_models):
        rmse = model["rmse"].loc[variable]
        axes.plot(
            [x for x in range(1, 11)],
            rmse.groupby(np.arange(len(rmse)) // 4).mean(),
            label=model["model_name"] + f" Future",
            linestyle="dashed",
            color=f"C{idx}",
        )

    # Add line style legend for past vs future
    past_line = mlines.Line2D(
        [], [], color="black", linestyle="-", label="U_T1"
    )
    future_line = mlines.Line2D(
        [], [], color="black", linestyle="--", label="U_T2"
    )
    axes.legend(
        handles=[past_line, future_line], loc="upper left", title="Test Set"
    )

    axes.set_title(f"{display_name} - RMSE Daily Average")
    axes.set_xlabel("Forecasting Time [Days]")
    axes.set_ylabel(f"RMSE")
    axes.grid(True)
    fig.tight_layout()
    fig.savefig(
        f"{plot_folder}/rmse_plots/daily_average_{figname}_{variable}_hist+future_together.png",
        dpi=400,
    )
    fig.savefig(
        f"{plot_folder}/rmse_plots/daily_average_{figname}_{variable}_hist+future_together.pdf",
        dpi=400,
    )
    plt.close()

# Plot R2

In [ ]:
import matplotlib.lines as mlines


figname = "r2_allmodels_future"

for display_name, unit, variable, plevel, _ in all_vars:

    fig, axes = plt.subplots(1, 1, figsize=(6, 4))
    for model in past_models:
        r2 = model["r2"].loc[variable]
        axes.plot(
            [x for x in range(1, 11)],
            r2.groupby(np.arange(len(r2)) // 4).mean(),
            label=model["model_name"] + f"",
        )

    legend1 = axes.legend(title="Models", loc="lower left")
    axes.add_artist(legend1)
    for idx, model in enumerate(future_models):
        r2 = model["r2"].loc[variable]
        axes.plot(
            [x for x in range(1, 11)],
            r2.groupby(np.arange(len(r2)) // 4).mean(),
            label=model["model_name"] + f" Future",
            linestyle="dashed",
            color=f"C{idx}",
        )

    # Add line style legend for past vs future
    past_line = mlines.Line2D([], [], color="black", linestyle="-", label="U_T1")
    future_line = mlines.Line2D(
        [], [], color="black", linestyle="--", label="U_T2"
    )
    axes.legend(
        handles=[past_line, future_line], loc="upper right", title="Test Set"
    )

    axes.set_title(f"{display_name} - R² Daily Average")
    axes.set_xlabel("Forecasting Time [Days]")
    axes.set_ylabel(f"R² score")
    axes.grid(True)
    fig.tight_layout()
    fig.savefig(
        f"{plot_folder}/r2_plots/daily_average_{figname}_{variable}_hist+future_together.png",
        dpi=400,
    )
    fig.savefig(
        f"{plot_folder}/r2_plots/daily_average_{figname}_{variable}_hist+future_together.pdf",
        dpi=400,
    )
    plt.close()

In [ ]:
# for display_name, unit, variable, plevel, _ in all_vars:

#     fig, axes = plt.subplots(1, 2, figsize=(20, 6))
#     for model in past_models:
#         axes[0].plot(
#             [x * 6 for x in range(1, 41)],
#             model["rmse"].loc[variable],
#             label=model["model_name"] + f"",
#         )
#         axes[0].set_title(f"{display_name} - RMSE - Test Period: 2011-2014")
#         axes[0].set_xlabel("Hours")
#         axes[0].set_ylabel(f"RMSE")
#         axes[0].legend()
#         axes[0].grid(True)
#     for model in future_models:
#         axes[1].plot(
#             [x * 6 for x in range(1, 41)],
#             model["rmse"].loc[variable],
#             label=model["model_name"] + f"",
#         )
#         axes[1].set_title(f"{display_name} - RMSE - Test Period: 2097-2100")
#         axes[1].set_xlabel("Hours")
#         axes[1].set_ylabel(f"RMSE")
#         axes[1].legend()
#         axes[1].grid(True)
#     utils.sync_axes(axes, sync_x=False)

#     fig.tight_layout()
#     fig.savefig(
#         f"{plot_folder}/rmse_plots/hourly/{figname}_{variable}_hist+future.png",
#         dpi=400,
#     )
#     fig.savefig(
#         f"{plot_folder}/rmse_plots/hourly/{figname}_{variable}_hist+future.pdf",
#         dpi=400,
#     )
#     plt.close()

In [ ]:
# for display_name, unit, variable, plevel, _ in all_vars:

#     fig, axes = plt.subplots(1, 2, figsize=(20, 6))
#     for model in past_models:
#         data = model["rmse"].loc[variable]
#         axes[0].plot(
#             [x for x in range(1, 11)],
#             data.groupby(np.arange(len(data)) // 4).mean(),
#             label=model["model_name"] + f"",
#         )
#         axes[0].set_title(
#             f"{display_name} - RMSE Daily Average - Test Period: 2011-2014"
#         )
#         axes[0].set_xlabel("Days")
#         axes[0].set_ylabel(f"RMSE")
#         axes[0].legend()
#         axes[0].grid(True)
#     for model in future_models:
#         data = model["rmse"].loc[variable]
#         axes[1].plot(
#             [x for x in range(1, 11)],
#             data.groupby(np.arange(len(data)) // 4).mean(),
#             label=model["model_name"] + f"",
#         )
#         axes[1].set_title(
#             f"{display_name} - RMSE Daily Average - Test Period: 2097-2100"
#         )
#         axes[1].set_xlabel("Days")
#         axes[1].set_ylabel(f"RMSE")
#         axes[1].legend()
#         axes[1].grid(True)

#     utils.sync_axes(axes, sync_x=False)
#     fig.tight_layout()
#     fig.savefig(
#         f"{plot_folder}/rmse_plots/daily/daily_average_{figname}_{variable}_hist+future.png",
#         dpi=400,
#     )
#     fig.savefig(
#         f"{plot_folder}/rmse_plots/daily/daily_average_{figname}_{variable}_hist+future.pdf",
#         dpi=400,
#     )
#     plt.close()